# NB04 – Média sozinha engana
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 4**

## A pergunta de negócio
A diretoria do e-commerce quer revisar a política de frete e pergunta:

> **Quanto custa um frete "normal"? Há cobranças fora do padrão? O frete acompanha o peso do produto?**

Para responder, precisamos olhar a **distribuição** dos valores, e não só a média.

**Dados:** itens e produtos da Olist (mesma base da Aula 3).

## 1. Carregar e juntar

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.float_format", "{:.2f}".format)

BASE = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/"
itens = pd.read_csv(BASE + "olist_itens.csv.gz")
produtos = pd.read_csv(BASE + "olist_produtos.csv.gz")
base = itens.merge(produtos, on="product_id", how="left")   # cada linha é um item vendido
print(base.shape)
base.head(3)

## 2. Classes de variáveis
Antes de escolher um gráfico ou uma medida, pergunte: **que tipo de variável é esta?**

| Classe | Subtipo | Exemplo nesta base | Medidas e gráficos |
|---|---|---|---|
| **Quantitativa** (número que se mede ou conta) | Contínua | `price`, `freight_value`, `product_weight_g` | média, mediana, desvio; histograma, box plot |
| | Discreta | `order_item_id` (posição do item no pedido) | contagens, média |
| **Qualitativa** (categoria) | Nominal (sem ordem) | `product_category_name`, `seller_id` | frequência; gráfico de barras |
| | Ordinal (com ordem) | nota de 1 a 5 (Aula 3); faixa de peso (seção 6) | frequência, mediana |

**Cuidado:** `seller_id` parece um código, mas é categoria. Calcular a "média do seller_id" não faz sentido.

## 3. A forma da distribuição: o histograma

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(base["freight_value"], bins=80, ax=ax[0])
ax[0].set_title("Frete por item (R$)")
sns.histplot(base["freight_value"], bins=80, ax=ax[1], log_scale=True)
ax[1].set_title("Frete por item, escala logarítmica")
plt.show()

A distribuição tem uma **cauda longa à direita**: a maioria dos fretes é baixa, mas alguns são muito altos.
Na escala logarítmica (cada marca vale 10 vezes a anterior), a forma fica mais simétrica e mais fácil de ler.

## 4. Medidas de posição: média, mediana e moda

In [ ]:
frete = base["freight_value"]
print(f"Média:   R$ {frete.mean():.2f}")
print(f"Mediana: R$ {frete.median():.2f}   (metade dos fretes está abaixo deste valor)")
print(f"Moda:    R$ {frete.mode()[0]:.2f}   (valor mais frequente)")

**Média > mediana** é o sinal de uma cauda à direita: poucos valores altos puxam a média para cima.
Para dizer "quanto custa um frete típico", a **mediana** representa melhor o cliente comum.

## 5. Dispersão, percentis, assimetria e curtose

In [ ]:
q1, q3 = frete.quantile([0.25, 0.75])
print(f"Desvio padrão: R$ {frete.std():.2f}")
print(f"Amplitude interquartil (IQR = Q3 − Q1): R$ {q3 - q1:.2f}   (o 'meio' de 50% dos fretes)")
print(f"Coeficiente de variação: {frete.std() / frete.mean():.0%}   (desvio relativo à média)")
print()
print(frete.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).rename("percentil"))

In [ ]:
print(f"Assimetria: {frete.skew():.2f}   (0 = simétrica; positiva = cauda à direita)")
print(f"Curtose:    {frete.kurt():.2f}   (0 = como a curva normal; alta = muitos valores extremos)")

**Como ler:**
- **Percentil 90:** 90% dos fretes custam até esse valor. Bom para definir uma regra ("frete acima do P99 vai para revisão").
- **Assimetria** alta e positiva confirma a cauda à direita vista no histograma.
- **Curtose** alta indica caudas pesadas: valores extremos são muito mais comuns do que numa curva normal. Médias e desvios ficam instáveis.

## 6. Reorganizar variáveis: faixas de peso e box plot
Transformar uma variável contínua em **faixas** (uma variável ordinal) ajuda a comparar grupos.

In [ ]:
base["faixa_peso"] = pd.cut(base["product_weight_g"], bins=[-1, 500, 2000, 10000, 50000],
                            labels=["até 0,5 kg", "0,5 a 2 kg", "2 a 10 kg", "acima de 10 kg"])
print(base["faixa_peso"].value_counts().sort_index())

plt.figure(figsize=(9, 4))
sns.boxplot(data=base, x="faixa_peso", y="freight_value", showfliers=False)
plt.title("Frete por faixa de peso (sem os pontos extremos)")
plt.ylabel("Frete (R$)"); plt.xlabel("")
plt.show()

**Como ler o box plot:** a caixa vai do Q1 ao Q3; a linha do meio é a mediana; as hastes vão até 1,5 × IQR. O que passa disso é marcado como *outlier* (aqui escondemos esses pontos para ver as caixas).

## 7. Outliers: erro, fraude ou exceção legítima?
Duas regras comuns:
- **Regra do IQR:** fora de [Q1 − 1,5 × IQR, Q3 + 1,5 × IQR].
- **Z-score:** mais de 3 desvios padrão longe da média. Funciona mal em distribuições assimétricas como esta.

In [ ]:
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
out_iqr = base[base["freight_value"] > lim_sup]
z = (frete - frete.mean()) / frete.std()
out_z = base[z.abs() > 3]
print(f"Limite superior pela regra do IQR: R$ {lim_sup:.2f}")
print(f"Outliers pelo IQR: {len(out_iqr)} ({len(out_iqr) / len(base):.1%} dos itens)")
print(f"Outliers pelo z-score: {len(out_z)} ({len(out_z) / len(base):.1%} dos itens)")

In [ ]:
# Os fretes mais altos: são erro, fraude ou exceção legítima?
base.sort_values("freight_value", ascending=False)[["price", "freight_value", "product_weight_g", "product_category_name"]].head(8)

**Pare e pense:** um frete de centenas de reais para um produto de 30 kg é um erro? E para um produto de 200 g?
Outlier é uma **pergunta**, não uma resposta. O limiar estatístico só diz onde olhar.

## 8. Depuração: valores faltantes e impossíveis

In [ ]:
print(base[["product_category_name", "product_weight_g"]].isna().sum())
print("Itens com peso igual a zero:", (base["product_weight_g"] == 0).sum())

Decisões possíveis, e o efeito de cada uma:
- **Categoria faltante:** criar a categoria `"sem_categoria"` (mantém os itens nas contas de faturamento).
- **Peso faltante ou zero:** excluir só das análises que usam peso; peso zero é impossível e indica erro de cadastro.

In [ ]:
base["product_category_name"] = base["product_category_name"].fillna("sem_categoria")
base_peso = base[base["product_weight_g"] > 0].dropna(subset=["product_weight_g"])
print("Itens com peso válido:", len(base_peso))

## 9. Correlação: o frete acompanha o peso?
A **correlação** mede se duas variáveis andam juntas, de −1 a +1.
- **Pearson:** mede relação **linear** e é sensível a valores extremos.
- **Spearman:** usa a **ordem** dos valores (postos); mede relação monotônica (sobe junto, mesmo que não em linha reta) e resiste a extremos.

In [ ]:
colunas = ["price", "freight_value", "product_weight_g"]
pearson = base_peso[colunas].corr(method="pearson")
spearman = base_peso[colunas].corr(method="spearman")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(pearson, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="RdBu_r", ax=ax[0]); ax[0].set_title("Pearson")
sns.heatmap(spearman, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="RdBu_r", ax=ax[1]); ax[1].set_title("Spearman")
plt.show()

In [ ]:
amostra = base_peso.sample(5000, random_state=42)
plt.figure(figsize=(7, 5))
sns.scatterplot(data=amostra, x="product_weight_g", y="freight_value", alpha=0.3, s=12)
plt.xscale("log"); plt.yscale("log")
plt.title("Frete × peso (amostra de 5.000 itens, escalas logarítmicas)")
plt.xlabel("Peso (g)"); plt.ylabel("Frete (R$)")
plt.show()

**Como ler:** o frete cresce com o peso, mas com muita dispersão: produtos do mesmo peso pagam fretes bem diferentes (a distância também importa, e não está nesta tabela).
Correlação **não é causalidade**: preço e peso também andam juntos, e produtos caros podem ter frete alto por outros motivos (seguro, embalagem). A Aula 5 trata disso.

---
## 10. Sua vez: exercícios individuais

**Exercício 1.** Calcule média, mediana e assimetria do **preço** (`price`). A distribuição é simétrica?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 2.** Qual o **percentil 99** do preço? Quantos itens ficam acima dele?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 3.** Quantos itens têm **frete maior que o preço** do produto? Que porcentagem isso representa? Isso é erro, fraude ou pode ser legítimo?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 4.** Calcule a **mediana do frete** em cada faixa de peso. Quanto o frete mediano de um item acima de 10 kg é maior que o de um item até 0,5 kg?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 5.** Compare as correlações de Pearson e de Spearman entre **preço e peso**. Por que elas são diferentes?

In [ ]:
# SEU CÓDIGO AQUI


---
## 11. Resposta à diretoria (e o que ela não diz)
- **Frete típico:** a mediana (cerca de R$ 16) representa melhor o cliente comum do que a média (cerca de R$ 20).
- **Fora do padrão:** pela regra do IQR, cerca de 10% dos itens têm frete acima de R$ 33. É muito para revisar um a um: vale focar numa faixa mais extrema (por exemplo, acima do P99, cerca de R$ 85) e cruzar com o peso.
- **Frete × peso:** relação positiva e moderada; o peso explica parte do frete, não tudo.

**Limitações:** não temos a distância (CEP) nesta tabela; preço e frete são por item, não por pedido; dados de 2016 a 2018.

---
## Erros comuns

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `KeyError: 'faixa_peso'` | A seção 6 não foi executada | Execute as células anteriores |
| `TypeError: ... category` ao somar faixas | Faixas são categorias, não números | Use `groupby` ou `value_counts` |
| Aviso `FutureWarning: observed=False` | Mudança de comportamento do pandas | Pode ignorar, ou use `observed=True` no `groupby` |
| Gráfico em branco | Faltou `plt.show()` ou a célula foi interrompida | Execute de novo |

## Plano B
Se o download falhar, carregue os arquivos `olist_itens.csv.gz` e `olist_produtos.csv.gz` pelo painel de arquivos do Colab e troque a linha `BASE = ...` por `BASE = ""`.